# Semana 3: Apuntes de la clase

**Valoración por flujo de caja descontado: FCFF y FCFE** (CFA L2, Equity: *Free Cash Flow Valuation*)

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JonathanRosasV/topicos-finanzas-upao/blob/main/03_dcf_fcff_fcfe/clase03_apuntes.ipynb)

Este notebook resume los conceptos que debiste llevarte de la clase, con los ejemplos numéricos ejecutables. Úsalo para repasar antes de la tarea y de las evaluaciones.

## Glosario de siglas de la semana

Antes de los conceptos, el idioma. Estas son las siglas que usamos esta semana; las de origen inglés se usan tal cual en la práctica profesional y en el examen CFA.

| Sigla | Significado | En pocas palabras |
|---|---|---|
| FCFF | *Free Cash Flow to the Firm* (flujo de caja libre de la firma) | Caja disponible para todos los financistas, accionistas y acreedores, tras operar e invertir. |
| FCFE | *Free Cash Flow to Equity* (flujo de caja libre del accionista) | Caja disponible solo para el accionista, después de servir la deuda. |
| EBIT | *Earnings Before Interest and Taxes* | Utilidad operativa, antes de intereses e impuestos; punto de partida favorito para proyectar. |
| NI | *Net Income* (utilidad neta) | La última línea del estado de resultados. |
| NCC | *Non-Cash Charges* (cargos que no son caja) | Partidas que restan utilidad sin mover caja, principalmente depreciación y amortización. |
| FCInv | *Fixed Capital Investment* (capex) | Inversión en activo fijo del periodo. |
| WCInv | *Working Capital Investment* | Incremento del capital de trabajo operativo; consume caja cuando crece. |
| CFO | *Cash Flow from Operations* (flujo de caja operativo) | La caja que generó la operación según el estado de flujos de efectivo. |
| Int | Gasto por intereses | Flujo para los acreedores; en el FCFF se devuelve después de impuestos: Int por (1 menos t). |
| EV | *Enterprise Value* (valor de la firma) | Lo que vale el negocio completo; menos deuda neta da el valor del equity. |
| VT | Valor Terminal | Valor de todos los flujos posteriores al horizonte de proyección; protagonista de la semana 4. |
| WACC | *Weighted Average Cost of Capital* | La tasa de descuento del FCFF (el FCFE se descuenta al Ke). |
| PC1 | Práctica Calificada 1 | Evaluación de hoy (semanas 1 a 3). |


## 1. La utilidad neta no es caja

La valoración descuenta **caja**, no utilidad contable. Cuatro ajustes separan una de otra: la depreciación (resta contable que no sale caja), el capex (salida de caja que no pasa por el estado de resultados), el incremento del capital de trabajo (vender al crédito genera utilidad hoy y caja después) y los intereses (flujo para un financista, no costo operativo del negocio).

## 2. Las definiciones centrales

**FCFF** (Free Cash Flow to the Firm): caja disponible para todos los financistas, accionistas y acreedores.

$$\text{FCFF} = \text{NI} + \text{NCC} + \text{Int}(1-t) - \text{FCInv} - \text{WCInv}$$

$$\text{FCFF} = \text{EBIT}(1-t) + \text{Dep} - \text{FCInv} - \text{WCInv}$$

$$\text{FCFF} = \text{CFO} + \text{Int}(1-t) - \text{FCInv}$$

**FCFE** (Free Cash Flow to Equity): caja disponible solo para el accionista.

$$\text{FCFE} = \text{FCFF} - \text{Int}(1-t) + \text{Endeudamiento neto}$$

El interés se devuelve **después de impuestos**: devolverlo completo sobreestima el FCFF exactamente en $\text{Int} \times t$ (trampa clásica de examen).

## 3. La regla de consistencia

| Flujo | Tasa | Resultado |
|---|---|---|
| FCFF | WACC | Valor de la firma (EV) |
| FCFE | $K_e$ | Valor del equity |

Del EV al equity se cruza el puente: $V_{equity} = EV - \text{Deuda neta}$, y entre el número de acciones se llega al valor por acción.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))
from utils.finanzas import (fcff_desde_ebit, fcff_desde_ni, fcff_desde_cfo,
                            fcfe_desde_fcff, valor_crecimiento_constante)

# Ejemplo integrador de la clase: EBIT 100, t 29.5%, Dep 30, capex 40,
# incremento de capital de trabajo 10, interes 19.5, endeudamiento neto +10
t = 0.295
f_ebit = fcff_desde_ebit(ebit=100, t=t, dep=30, fcinv=40, wcinv=10)

ni = (100 - 19.5) * (1 - t)                 # utilidad neta = 56.75
f_ni = fcff_desde_ni(ni=ni, ncc=30, interes=19.5, t=t, fcinv=40, wcinv=10)

cfo = ni + 30 - 10                          # CFO = NI + Dep - WCInv = 76.75
f_cfo = fcff_desde_cfo(cfo=cfo, interes=19.5, t=t, fcinv=40)

print(f"FCFF desde EBIT = {f_ebit:.2f}")
print(f"FCFF desde NI   = {f_ni:.2f}")
print(f"FCFF desde CFO  = {f_cfo:.2f}")
print("Las tres rutas coinciden:", round(f_ebit, 10) == round(f_ni, 10) == round(f_cfo, 10))

fcfe = fcfe_desde_fcff(f_ebit, interes=19.5, t=t, endeudamiento_neto=10)
print(f"FCFE = {fcfe:.2f}")

Salida esperada: FCFF = 50.50 por las tres rutas y FCFE = 46.75. Que las rutas coincidan es un control de calidad gratuito: si no coinciden, hay un error en los insumos.

## 4. Modelos de valoración

**Una etapa** (crecimiento constante a perpetuidad, requiere $g < r$):

$$V_0^{firma} = \frac{\text{FCFF}_1}{\text{WACC} - g} \qquad\qquad V_0^{equity} = \frac{\text{FCFE}_1}{K_e - g}$$

**Dos etapas**: proyección explícita de $n$ años mas valor terminal $VT_n = \frac{\text{FCFF}_{n+1}}{\text{WACC} - g}$, todo descontado a hoy. El valor terminal suele concentrar 60 a 80 por ciento del valor total (semana 4).

In [ ]:
# Valoracion del ejemplo con g = 4%, WACC = 9.14%, Ke = 11.1%, deuda = 300
g, WACC, Ke, D = 0.04, 0.0914, 0.111, 300

EV = valor_crecimiento_constante(f_ebit * (1 + g), WACC, g)
eq_fcff = EV - D
eq_fcfe = valor_crecimiento_constante(fcfe * (1 + g), Ke, g)

print(f"EV (ruta FCFF)          = {EV:,.1f}")
print(f"Equity via FCFF (EV - D) = {eq_fcff:,.1f}")
print(f"Equity via FCFE          = {eq_fcfe:,.1f}")
print(f"Valor de mercado         = 700.0")

Salida esperada: EV = 1,021.8, equity por FCFF = 721.8 y por FCFE = 684.8, contra 700 de mercado. Las dos rutas solo coinciden exactamente si los supuestos de deuda son perfectamente consistentes entre sí. Con estimaciones tan cerca del precio, el veredicto profesional honesto es "bien valorada, sujeto a análisis de sensibilidad" (semana 4). Recuerda la semana 1: la diferencia entre tu estimación y el precio mezcla mispricing verdadero y error de estimación.

## 5. Sensibilidad del denominador

En el modelo de una etapa el denominador $r - g$ es pequeño, así que cambios menores en $g$ o en la tasa mueven el valor de forma desproporcionada.

In [ ]:
# Subir g de 4% a 5% con WACC 9.14%: el valor sube cerca de 25%
# (aqui el flujo del anio 1 tambien crece con el nuevo g; con FCFF_1 fijo la subida es 24.2%)
V_g4 = valor_crecimiento_constante(f_ebit * 1.04, WACC, 0.04)
V_g5 = valor_crecimiento_constante(f_ebit * 1.05, WACC, 0.05)
print(f"V(g=4%) = {V_g4:,.1f} | V(g=5%) = {V_g5:,.1f} | variacion = {V_g5/V_g4 - 1:.1%}")

## 6. ¿FCFF o FCFE?

FCFF cuando la estructura de capital está cambiando (el WACC es más estable que el $K_e$ en ese escenario) o cuando el FCFE es negativo. FCFE cuando el apalancamiento es estable y se quiere el equity directo. Y FCF en general por encima de modelos de dividendos cuando los dividendos no reflejan la capacidad de pago: el FCFE es lo que la empresa **podría** repartir; el dividendo es lo que **decide** repartir.

## 7. Respuestas a los ítems de la clase

**1. B.** Devolver el interés completo en vez de $\text{Int}(1-t)$ sobreestima el FCFF en $\text{Int} \times t$: se está devolviendo también el escudo tributario, que la firma no entrega al acreedor.

**2. B.** $\text{FCFE} = 80 - 20(1 - 0.30) + 15 = 80 - 14 + 15 = 81$.

**3. B.** Con estructura de capital cambiando (reducción agresiva de deuda), FCFF descontado al WACC y luego el puente al equity. La opción C viola la regla de consistencia.

**4. B.** "Manteniendo todo lo demás constante" significa que el flujo del año 1 no cambia: $V$ pasa de $\frac{FCFF_1}{0.09-0.04}$ a $\frac{FCFF_1}{0.09-0.05}$, y la razón es $\frac{0.05}{0.04} = 1.25$, una subida de exactamente 25 por ciento. Si además dejas que el flujo del año 1 crezca con el nuevo $g$ (de $1.04\,FCFF_0$ a $1.05\,FCFF_0$), la razón sube a 1.26. En ambos casos la lección es la misma: un punto más de $g$ no sube el valor 1 por ciento, porque el denominador manda.

## 8. Lista de verificación para la PC1

Sabes de esta semana si puedes: construir el FCFF por las tres rutas y explicar por qué coinciden; pasar de FCFF a FCFE; enunciar la regla de consistencia completa (flujo, tasa, resultado); valorar con la perpetuidad creciente y cruzar el puente EV a equity; y detectar los seis errores comunes de la última lámina.